In [ ]:
pip install opencv-python numpy segmentation-models-pytorch scikit-learn tqdm rasterio


In [ ]:
import os
import glob
import cv2
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support, accuracy_score
from tqdm import tqdm
import rasterio
from rasterio.windows import Window

In [3]:
# ==========================================
# 1. CONFIGURATION & SETUP
# ==========================================
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
BATCH_SIZE = 8
EPOCHS = 10
LR = 0.0001
IMG_SIZE = 512
NUM_CLASSES = 8

# Define Paths
# Assuming structure: Theme 1/Final Model Training + Predictions/model_training+predictions.ipynb
# We need to go up one level to access processed_tiles
BASE_DIR = "../" 
TRAIN_IMG_DIR = os.path.join(BASE_DIR, "processed_tiles/images")
TRAIN_MASK_DIR = os.path.join(BASE_DIR, "processed_tiles/masks")

# Class Definitions
CLASS_MAP = {
    0: "Background",
    1: "Waterbody",
    2: "Road",
    3: "RCC Roof",
    4: "Tiled Roof",
    5: "Tin Roof",
    6: "Other Roof",
    7: "Utility"
}

print(f"✅ Running on device: {DEVICE}")

# ==========================================
# 2. DATASET CLASS
# ==========================================
class DroneDataset(Dataset):
    def __init__(self, image_paths, mask_paths=None, transform=None):
        self.image_paths = image_paths
        self.mask_paths = mask_paths
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        # Read Image
        img_path = self.image_paths[idx]
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        # Normalize (0-1) and float
        image = image.astype('float32') / 255.0
        image = np.transpose(image, (2, 0, 1)) # HWC -> CHW

        if self.mask_paths:
            mask_path = self.mask_paths[idx]
            mask = cv2.imread(mask_path, 0) # Read as grayscale
            # Ensure mask is int64 for CrossEntropy
            mask = mask.astype('int64') 
            return torch.tensor(image), torch.tensor(mask)
        
        return torch.tensor(image)

# ==========================================
# 3. PREPARE DATA
# ==========================================
all_images = sorted(glob.glob(os.path.join(TRAIN_IMG_DIR, "**/*.png"), recursive=True))
all_masks = sorted(glob.glob(os.path.join(TRAIN_MASK_DIR, "**/*.png"), recursive=True))

print(f"Found {len(all_images)} training images and {len(all_masks)} masks.")

# Validation Split (80/20)
train_imgs, val_imgs, train_masks, val_masks = train_test_split(all_images, all_masks, test_size=0.2, random_state=42)

train_dataset = DroneDataset(train_imgs, train_masks)
val_dataset = DroneDataset(val_imgs, val_masks)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

# ==========================================
# 4. MODEL SETUP (UNET)
# ==========================================
model = smp.Unet(
    encoder_name="resnet34",        
    encoder_weights="imagenet",     
    in_channels=3,                  
    classes=NUM_CLASSES,            
)
model.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

# ==========================================
# 5. TRAINING LOOP
# ==========================================
print("🚀 Starting Training...")

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0
    
    for images, masks in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        images = images.to(DEVICE)
        masks = masks.to(DEVICE)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        
    print(f"Epoch {epoch+1} Loss: {train_loss/len(train_loader):.4f}")

# ==========================================
# 6. EXTENDED VALIDATION & METRICS
# ==========================================
print("\n📊 Calculating Detailed Validation Metrics...")
model.eval()

# Accumulate ground truth and predictions for global metrics
all_preds = []
all_targets = []

with torch.no_grad():
    for images, masks in tqdm(val_loader, desc="Validating"):
        images = images.to(DEVICE)
        outputs = model(images)
        preds = torch.argmax(outputs, dim=1).cpu().numpy()
        targets = masks.cpu().numpy()
        
        all_preds.append(preds.flatten())
        all_targets.append(targets.flatten())

# Concatenate all batches
all_preds = np.concatenate(all_preds)
all_targets = np.concatenate(all_targets)

# Compute Confusion Matrix
cm = confusion_matrix(all_targets, all_preds, labels=range(NUM_CLASSES))

# 1. Pixel Accuracy
pixel_acc = np.trace(cm) / np.sum(cm)

# 2. Class-wise IoU & mIoU
intersection = np.diag(cm)
union = np.sum(cm, axis=0) + np.sum(cm, axis=1) - intersection
iou_per_class = np.divide(intersection, union, out=np.zeros_like(intersection, dtype=float), where=union!=0)
mean_iou = np.mean(iou_per_class)

# 3. Class-wise Pixel Accuracy & mPA
class_total = np.sum(cm, axis=1)
pixel_acc_per_class = np.divide(intersection, class_total, out=np.zeros_like(intersection, dtype=float), where=class_total!=0)
mean_pixel_acc = np.mean(pixel_acc_per_class)

# 4. Precision, Recall, F1
# using sklearn for convenience on the flattened arrays (weighted or macro)
precision_cls, recall_cls, f1_cls, _ = precision_recall_fscore_support(all_targets, all_preds, labels=range(NUM_CLASSES), average=None, zero_division=0)
macro_f1 = np.mean(f1_cls)

print("\n" + "="*40)
print(f"🏆 VALIDATION RESULTS")
print("="*40)
print(f"✅ Overall Pixel Accuracy:  {pixel_acc:.4f}")
print(f"✅ Mean Pixel Accuracy (mPA): {mean_pixel_acc:.4f}")
print(f"✅ Mean IoU (mIoU):         {mean_iou:.4f}")
print(f"✅ Macro F1 Score:          {macro_f1:.4f}")
print("-" * 40)
print(f"{'Class':<15} | {'IoU':<8} | {'Precision':<10} | {'Recall':<10} | {'F1':<10}")
print("-" * 65)
for i in range(NUM_CLASSES):
    print(f"{CLASS_MAP[i]:<15} | {iou_per_class[i]:.4f}   | {precision_cls[i]:.4f}     | {recall_cls[i]:.4f}     | {f1_cls[i]:.4f}")
print("="*40)

# ==========================================
# 7. SAVE MODEL
# ==========================================
model_save_path = "model_round1.pth"
torch.save(model.state_dict(), model_save_path)
torch.save(model, "full_model_round1.pth")
print(f"\n💾 Model weights saved to: {os.path.abspath(model_save_path)}")

✅ Running on device: cuda
Found 7123 training images and 7123 masks.
🚀 Starting Training...


Epoch 1/10: 100%|████████████████████████████████████████████████████████████████████| 713/713 [07:30<00:00,  1.58it/s]


Epoch 1 Loss: 0.9604


Epoch 2/10: 100%|████████████████████████████████████████████████████████████████████| 713/713 [07:12<00:00,  1.65it/s]


Epoch 2 Loss: 0.5413


...

Epoch 9 Loss: 0.2007


Epoch 10/10: 100%|███████████████████████████████████████████████████████████████████| 713/713 [07:20<00:00,  1.62it/s]


Epoch 10 Loss: 0.1691

📊 Calculating Detailed Validation Metrics...


Validating: 100%|████████████████████████████████████████████████████████████████████| 179/179 [00:52<00:00,  3.44it/s]



🏆 VALIDATION RESULTS
✅ Overall Pixel Accuracy:  0.9042
✅ Mean Pixel Accuracy (mPA): 0.6539
✅ Mean IoU (mIoU):         0.5732
✅ Macro F1 Score:          0.6761
----------------------------------------
Class           | IoU      | Precision  | Recall     | F1        
-----------------------------------------------------------------
Background      | 0.8789   | 0.9182     | 0.9536     | 0.9356
Waterbody       | 0.7087   | 0.8670     | 0.7952     | 0.8295
Road            | 0.6238   | 0.8736     | 0.6856     | 0.7683
RCC Roof        | 0.7941   | 0.8874     | 0.8831     | 0.8852
Tiled Roof      | 0.5278   | 0.6607     | 0.7241     | 0.6909
Tin Roof        | 0.7880   | 0.8854     | 0.8774     | 0.8814
Other Roof      | 0.2642   | 0.6311     | 0.3124     | 0.4180
Utility         | 0.0000   | 0.0000     | 0.0000     | 0.0000

💾 Model weights saved to: C:\Users\shaur\Desktop\ML\Projects\National GEOAI Hackathon\Theme 1\Try 5\model_round1.pth


In [ ]:
import os
import glob
import cv2
import numpy as np
import torch
import rasterio
from rasterio.windows import Window
from tqdm import tqdm

# ==========================================
# 1. INFERENCE SETUP
# ==========================================
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
IMG_SIZE = 512
NUM_CLASSES = 8
BASE_DIR = "../" # Relative path to Theme 1 root
TEST_DATA_DIR = os.path.join(BASE_DIR, "test_data")

# Output Directories
PRED_REPORT_DIR = "test_predictions_report"
PRED_OVERLAP_DIR = "test_predictions_overlap"
os.makedirs(PRED_REPORT_DIR, exist_ok=True)
os.makedirs(PRED_OVERLAP_DIR, exist_ok=True)

#Files to EXCLUDE from this run (because they are too big and crash RAM)
# These will be handled by Cell 2
EXCLUDE_FILES = [
    "BUTTAR SIVIYA _AMRITSAR_37810_ORTHO.tif",
    "DIWANA_BARNALA_40082_ORTHO.tif",
    "KARTARPUR_AMRITSAR_37842_ORTHO.tif"
]

# Required Color Map (RGB)
COLOR_MAP_RGB = {
    0: [0, 0, 0],       # Black (Background)
    1: [0, 120, 255],   # Bright Blue (Water)
    2: [255, 165, 0],   # Orange (Road)
    3: [0, 255, 0],     # Green (RCC Roof)
    4: [255, 0, 255],   # Magenta (Tiled Roof)
    5: [0, 255, 255],   # Cyan (Tin Roof)
    6: [255, 0, 0],     # Red (Other Roof)
    7: [255, 255, 0]    # Yellow (Utility)
}

# Load Model (Ensure it's in memory or reload)
if 'model' not in globals():
    print("Loading model from disk...")
    model = torch.load("full_model_round1.pth")
    model.to(DEVICE)
model.eval()

# ==========================================
# 2. HELPER FUNCTIONS
# ==========================================

def colorize_mask(mask_2d):
    """
    Converts a single channel (H,W) mask with values 0-7 
    into a 3-channel RGB image (H,W,3) using COLOR_MAP_RGB.
    """
    h, w = mask_2d.shape
    color_mask = np.zeros((h, w, 3), dtype=np.uint8)
    
    for cls_idx, color in COLOR_MAP_RGB.items():
        # Mask where class matches
        idx = mask_2d == cls_idx
        color_mask[idx] = color
        
    return color_mask

def blend_images(original_rgb, color_mask_rgb, alpha=0.6):
    """
    Overlays color mask on original image.
    """
    # Ensure sizes match
    if original_rgb.shape != color_mask_rgb.shape:
        color_mask_rgb = cv2.resize(color_mask_rgb, (original_rgb.shape[1], original_rgb.shape[0]), interpolation=cv2.INTER_NEAREST)
        
    # Standard weighting: alpha * src1 + beta * src2 + gamma
    beta = 1.0 - alpha
    blended = cv2.addWeighted(original_rgb, alpha, color_mask_rgb, beta, 0.0)
    return blended

def predict_and_save(model, tif_path):
    """
    Runs sliding window inference, stitches result, 
    saves color-coded mask and overlay.
    """
    filename = os.path.basename(tif_path).replace(".tif", "")
    
    with rasterio.open(tif_path) as src:
        h, w = src.height, src.width
        # Prepare full mask container (single channel for prediction logic)
        full_pred_mask = np.zeros((h, w), dtype=np.uint8)
        
        # We also need the Full RGB Image for the overlay
        print(f"Reading source image: {filename} ({w}x{h})...")
        full_src_img = src.read([1, 2, 3]) # Read RGB
        # Rasterio reads as (C, H, W), convert to (H, W, C)
        full_src_img = np.transpose(full_src_img, (1, 2, 0))

        # --- SLIDING WINDOW INFERENCE ---
        for i in tqdm(range(0, h, IMG_SIZE), desc="Processing Tiles", leave=False):
            for j in range(0, w, IMG_SIZE):
                window = Window(j, i, min(IMG_SIZE, w-j), min(IMG_SIZE, h-i))
                
                # Read specific window for inference
                img_patch = src.read(window=window)
                
                # Take only RGB if RGBA
                if img_patch.shape[0] == 4:
                    img_patch = img_patch[:3, :, :]
                
                # Handle edge cases where patch is smaller than IMG_SIZE
                _, curr_h, curr_w = img_patch.shape
                pad_patch = np.zeros((3, IMG_SIZE, IMG_SIZE), dtype=np.float32)
                pad_patch[:, :curr_h, :curr_w] = img_patch / 255.0
                
                input_tensor = torch.from_numpy(pad_patch).unsqueeze(0).to(DEVICE)
                
                with torch.no_grad():
                    output = model(input_tensor)
                    pred = torch.argmax(output, dim=1).cpu().numpy()[0]
                
                # Assign back to full mask
                full_pred_mask[i:i+curr_h, j:j+curr_w] = pred[:curr_h, :curr_w]

    # --- POST PROCESSING ---
    print(f"Generating outputs for {filename}...")
    
    # 1. Colorize Prediction
    color_pred = colorize_mask(full_pred_mask)
    
    # 2. Create Overlay
    overlay_img = blend_images(full_src_img, color_pred, alpha=0.6)
    
    # --- SAVE FILES ---
    
    # Save Color Mask (RGB) -> Convert to BGR for OpenCV saving
    save_path_report = os.path.join(PRED_REPORT_DIR, f"{filename}_prediction.png")
    cv2.imwrite(save_path_report, cv2.cvtColor(color_pred, cv2.COLOR_RGB2BGR))
    
    # Save Overlap (RGB) -> Convert to BGR for OpenCV saving
    save_path_overlap = os.path.join(PRED_OVERLAP_DIR, f"{filename}_overlay.png")
    cv2.imwrite(save_path_overlap, cv2.cvtColor(overlay_img, cv2.COLOR_RGB2BGR))

# ==========================================
# 3. RUN ON TEST DATA
# ==========================================
all_tifs = sorted(glob.glob(os.path.join(TEST_DATA_DIR, "**/*.tif"), recursive=True))

# ---------------------------------------------------------
# FILTER LOGIC: Keep only files NOT in the exclusion list
# ---------------------------------------------------------
test_tifs = [f for f in all_tifs if os.path.basename(f) not in EXCLUDE_FILES]

print(f"\n🔍 Found {len(all_tifs)} total files.")
print(f"⚠️ Excluding {len(EXCLUDE_FILES)} large files (to be handled in next cell).")
print(f"🚀 Processing remaining {len(test_tifs)} standard files...\n")

for tif_file in test_tifs:
    predict_and_save(model, tif_file)

print(f"\n✅ Standard Tasks Completed.")

In [6]:
import gc

# ==========================================
# 1. SETUP & CONFIGURATION
# ==========================================
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
IMG_SIZE = 512
BASE_DIR = "../" 
TEST_DATA_DIR = os.path.join(BASE_DIR, "test_data")
PRED_REPORT_DIR = "test_predictions_report"
PRED_OVERLAP_DIR = "test_predictions_overlap"

os.makedirs(PRED_REPORT_DIR, exist_ok=True)
os.makedirs(PRED_OVERLAP_DIR, exist_ok=True)

# The 3 specific files that failed
TARGET_FILES = [
    "BUTTAR SIVIYA _AMRITSAR_37810_ORTHO.tif",
    "DIWANA_BARNALA_40082_ORTHO.tif",
    "KARTARPUR_AMRITSAR_37842_ORTHO.tif"
]

# Palette in BGR format (OpenCV uses BGR, not RGB)
# This prevents us from needing cv2.cvtColor later (saving memory)
COLOR_MAP_BGR = {
    0: [0, 0, 0],       # Black
    1: [255, 120, 0],   # Blue
    2: [0, 165, 255],   # Orange
    3: [0, 255, 0],     # Green
    4: [255, 0, 255],   # Magenta
    5: [255, 255, 0],   # Cyan
    6: [0, 0, 255],     # Red
    7: [0, 255, 255]    # Yellow
}

# ==========================================
# 2. MEMORY OPTIMIZED FUNCTIONS
# ==========================================
def colorize_mask_to_bgr(mask_2d):
    """Creates BGR color mask using a lookup table."""
    palette = np.zeros((256, 3), dtype=np.uint8)
    for cls_idx, color in COLOR_MAP_BGR.items():
        palette[cls_idx] = color
    return palette[mask_2d]

def blend_in_place(base_image, overlay_image, alpha=0.6):
    """
    Blends overlay_image ONTO base_image in chunks.
    Overwrites base_image with the result to save RAM.
    """
    beta = 1.0 - alpha
    h, w, c = base_image.shape
    step = 4096 # Process in 4k strips to keep memory low

    for i in tqdm(range(0, h, step), desc="  > Blending Chunks", leave=False):
        row_end = min(i + step, h)
        
        # Extract slices (Views, no copy)
        base_slice = base_image[i:row_end]
        overlay_slice = overlay_image[i:row_end]
        
        # Blend in place
        cv2.addWeighted(base_slice, alpha, overlay_slice, beta, 0.0, dst=base_slice)
        
    return base_image

# ==========================================
# 3. PROCESSING LOOP
# ==========================================
# Reload model just in case
if 'model' not in globals():
    model = torch.load("full_model_round1.pth")
    model.to(DEVICE)
model.eval()

# Filter files
all_tifs = sorted(glob.glob(os.path.join(TEST_DATA_DIR, "**/*.tif"), recursive=True))
target_paths = [p for p in all_tifs if os.path.basename(p) in TARGET_FILES]

print(f"🔍 Processing remaining {len(target_paths)} files...")

for tif_path in target_paths:
    filename = os.path.basename(tif_path).replace(".tif", "")
    print(f"\nProcessing: {filename} ...")
    
    # 1. Clean RAM
    gc.collect()
    torch.cuda.empty_cache()

    with rasterio.open(tif_path) as src:
        h, w = src.height, src.width
        
        # A. PREDICTION
        full_pred_mask = np.zeros((h, w), dtype=np.uint8)
        
        for i in tqdm(range(0, h, IMG_SIZE), desc="  > Inference", leave=False):
            for j in range(0, w, IMG_SIZE):
                window = Window(j, i, min(IMG_SIZE, w-j), min(IMG_SIZE, h-i))
                img_patch = src.read(window=window)
                if img_patch.shape[0] == 4: img_patch = img_patch[:3]
                
                _, curr_h, curr_w = img_patch.shape
                pad_patch = np.zeros((3, IMG_SIZE, IMG_SIZE), dtype=np.float32)
                pad_patch[:, :curr_h, :curr_w] = img_patch / 255.0
                
                input_tensor = torch.from_numpy(pad_patch).unsqueeze(0).to(DEVICE)
                with torch.no_grad():
                    output = model(input_tensor)
                    pred = torch.argmax(output, dim=1).cpu().numpy()[0]
                
                full_pred_mask[i:i+curr_h, j:j+curr_w] = pred[:curr_h, :curr_w]

        # B. SAVE PREDICTION (COLOR MASK)
        print("  > Saving Prediction Mask...")
        # Create BGR mask directly
        bgr_pred = colorize_mask_to_bgr(full_pred_mask)
        # Release raw mask to free memory
        del full_pred_mask 
        
        save_path_report = os.path.join(PRED_REPORT_DIR, f"{filename}_prediction.png")
        cv2.imwrite(save_path_report, bgr_pred)

        # C. CREATE OVERLAY (If RAM permits)
        print("  > Creating Overlay...")
        try:
            # Read image as RGB, then swap to BGR manually to avoid cvtColor copy
            # Reading slice-by-slice might be needed if this fails, but usually 5GB is ok
            full_src_img = src.read([1, 2, 3]) 
            full_src_img = np.transpose(full_src_img, (1, 2, 0)) # HWC
            # Convert RGB to BGR in-place (view)
            full_src_img = full_src_img[..., ::-1].copy() # Copy ensures contiguous array for OpenCV
            
            # Blend In-Place (Overwrites full_src_img to save RAM)
            blend_in_place(full_src_img, bgr_pred, alpha=0.6)
            
            save_path_overlap = os.path.join(PRED_OVERLAP_DIR, f"{filename}_overlay.png")
            cv2.imwrite(save_path_overlap, full_src_img)
            
            del full_src_img
        except MemoryError:
            print("  ❌ Skipping Overlay for this file (Not enough RAM). Prediction mask was saved.")
        
        del bgr_pred
        gc.collect()

print("\n✅ Processing Complete.")

🔍 Processing remaining 3 files...

Processing: BUTTAR SIVIYA _AMRITSAR_37810_ORTHO ...


  > Saving Prediction Mask...
  > Creating Overlay...



Processing: DIWANA_BARNALA_40082_ORTHO ...


  > Saving Prediction Mask...
  > Creating Overlay...



Processing: KARTARPUR_AMRITSAR_37842_ORTHO ...


  > Saving Prediction Mask...
  > Creating Overlay...



✅ Processing Complete.
